In [11]:
import os
import librosa
import soundfile
import numpy as np
import pandas as pd
import sys
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import kagglehub
import warnings
import logging
warnings.filterwarnings("ignore")

# --- 1. SET UP LOGGING ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [18]:
# --- 2. DOWNLOAD THE RAVDESS DATASET ---
def download_ravdess_dataset():
    """
    Downloads the RAVDESS dataset from Kaggle Hub, handling potential errors.
    
    Returns:
        str: The path to the downloaded and extracted dataset directory.
    """
    logging.info("Downloading RAVDESS dataset from Kaggle...")
    try:
        # This downloads the dataset and returns the path to the extracted files.
        # The dataset contains a main directory 'audio_speech_actors_01-24'.
        path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
        dataset_path = os.path.join(path, "audio_speech_actors_01-24")
        logging.info(f"Dataset downloaded to: {dataset_path}")

        # Check if the expected directory exists.
        if not os.path.isdir(dataset_path):
            raise FileNotFoundError(f"Could not find dataset directory at {dataset_path}")
        
        return dataset_path
    except Exception as e:
        logging.error(f"Failed to download dataset from Kaggle: {e}")
        sys.exit(1)

In [6]:
# --- 3. FEATURE EXTRACTION ---
def extract_features(file_path, n_mfcc=40):
    """
    Extracts Mel-frequency cepstral coefficients (MFCCs) from an audio file.
    
    Args:
        file_path (str): The path to the audio file.
        n_mfcc (int): The number of MFCCs to extract.
        
    Returns:
        np.array: A 2D numpy array of MFCCs.
    """
    try:
        # Load the audio file
        y, sr = librosa.load(file_path, duration=3, offset=0.5, sr=22050)
        # Extract MFCCs
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc).T, axis=0)
        return mfccs
    except Exception as e:
        logging.error(f"Error encountered while parsing file: {file_path}. Error: {e}")
        return None



In [14]:
# --- 4. DATA PREPARATION ---
def load_data(dataset_path):
    """
    Loads all audio files from the dataset, extracts features, and creates labels.
    This version uses a pandas DataFrame for easier data management.
    
    Args:
        dataset_path (str): The path to the root of the dataset.
        
    Returns:
        tuple: A tuple containing features (numpy array) and labels (numpy array).
    """
    audio_paths = []
    labels = []
    
    # Mapping of emotion codes to labels as per the RAVDESS documentation
    emotion_map = {
        1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad',
        5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'
    }
    
    logging.info("Starting to collect file paths and labels...")
    
    # Iterate through each actor directory
    for subdir, _, files in os.walk(dataset_path):
        for file in files:
            # Check if the file is a WAV file
            if file.endswith('.wav'):
                file_path = os.path.join(subdir, file)
                # Filename format: '03-01-06-02-01-02-09.wav'
                # The emotion code is the third part of the filename.
                part = file.split('.')[0].split('-')
                emotion_code = int(part[2])
                
                # Only include files with a valid emotion code
                if emotion_code in emotion_map:
                    audio_paths.append(file_path)
                    labels.append(emotion_map[emotion_code])

    logging.info(f"Finished collecting paths. Total samples: {len(audio_paths)}")
    
    # Create a DataFrame for better data handling and visualization
    df = pd.DataFrame({"path": audio_paths, "label": labels})
    
    logging.info("\nDataFrame created successfully:")
    logging.info(df.head())
    
    # Now, extract features from the collected paths
    features = []
    
    logging.info("Starting feature extraction from audio files...")
    
    for path in df['path']:
        mfccs = extract_features(path)
        if mfccs is not None:
            features.append(mfccs)
    
    logging.info(f"Finished feature extraction. Total samples with features: {len(features)}")
    
    return np.array(features), np.array(df['label'][:len(features)])


In [19]:
    
# 1. Download the dataset
dataset_dir = download_ravdess_dataset()
    
# 2. Load and prepare the data
X, y = load_data(dataset_dir)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    

2025-08-14 21:41:00,464 - INFO - Downloading RAVDESS dataset from Kaggle...
2025-08-14 21:41:01,416 - INFO - Dataset downloaded to: C:\Users\ASUS\.cache\kagglehub\datasets\uwrfkaggler\ravdess-emotional-speech-audio\versions\1\audio_speech_actors_01-24
2025-08-14 21:41:01,417 - INFO - Starting to collect file paths and labels...
2025-08-14 21:41:01,500 - INFO - Finished collecting paths. Total samples: 1440
2025-08-14 21:41:01,500 - INFO - 
DataFrame created successfully:
2025-08-14 21:41:01,500 - INFO -                                                 path    label
0  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
1  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
2  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
3  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...  neutral
4  C:\Users\ASUS\.cache\kagglehub\datasets\uwrfka...     calm
2025-08-14 21:41:01,500 - INFO - Starting feature extraction from audio files...
2025-08-14 21:41:26,729 - INFO - Finis

In [21]:
# 3. Standardize the data
# It's good practice to scale features for MLP models
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
# 4. Create and train the MLPClassifier model
logging.info("Creating and training the MLPClassifier model...")

# Initialize the MLPClassifier with desired parameters
mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64), # Similar to the previous Keras model architecture
    activation='relu',
    solver='adam',
    max_iter=500, # Increased iterations for better convergence
    verbose=True,
    random_state=42
)

# Train the model
mlp_model.fit(X_train_scaled, y_train)

logging.info("Model training completed.")


2025-08-14 21:42:19,326 - INFO - Creating and training the MLPClassifier model...


Iteration 1, loss = 2.09308934
Iteration 2, loss = 1.89084878
Iteration 3, loss = 1.76267261
Iteration 4, loss = 1.64383314
Iteration 5, loss = 1.54225600
Iteration 6, loss = 1.44327181
Iteration 7, loss = 1.34273784
Iteration 8, loss = 1.25249194
Iteration 9, loss = 1.16506794
Iteration 10, loss = 1.07921689
Iteration 11, loss = 1.00284208
Iteration 12, loss = 0.92327036
Iteration 13, loss = 0.86331470
Iteration 14, loss = 0.79620696
Iteration 15, loss = 0.73575162
Iteration 16, loss = 0.68062102
Iteration 17, loss = 0.63461102
Iteration 18, loss = 0.58142441
Iteration 19, loss = 0.53704878
Iteration 20, loss = 0.49277192
Iteration 21, loss = 0.45495406
Iteration 22, loss = 0.41760596
Iteration 23, loss = 0.38449290
Iteration 24, loss = 0.35591412
Iteration 25, loss = 0.32350212
Iteration 26, loss = 0.30071079
Iteration 27, loss = 0.27685237
Iteration 28, loss = 0.25174089
Iteration 29, loss = 0.22515519
Iteration 30, loss = 0.20733449
Iteration 31, loss = 0.18924087
Iteration 32, los

2025-08-14 21:42:22,891 - INFO - Model training completed.


Iteration 120, loss = 0.00340365
Iteration 121, loss = 0.00333001
Iteration 122, loss = 0.00327315
Iteration 123, loss = 0.00321266
Iteration 124, loss = 0.00313168
Iteration 125, loss = 0.00308556
Iteration 126, loss = 0.00301228
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.


In [23]:
# 5. Evaluate the model
logging.info("Evaluating the model on the test data...")
y_pred = mlp_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy*100:.2f}%")


2025-08-14 21:42:37,764 - INFO - Evaluating the model on the test data...



Test Accuracy: 68.06%


In [24]:
# --- 6. SAVE THE TRAINED MODEL AND SCALER ---
import joblib

# Make sure to import joblib at the top of your script
# from joblib import dump, load

# Save the trained model and scaler to disk
model_filename = "mlp_emotion_model.joblib"
scaler_filename = "scaler.joblib"

joblib.dump(mlp_model, model_filename)
joblib.dump(scaler, scaler_filename)

logging.info(f"Model saved to {model_filename}")
logging.info(f"Scaler saved to {scaler_filename}")

2025-08-14 21:43:38,731 - INFO - Model saved to mlp_emotion_model.joblib
2025-08-14 21:43:38,733 - INFO - Scaler saved to scaler.joblib
